In [18]:
text ="The inference engine processes requests by loading model weights into GPU memory"

In [19]:
#1 token per character
tokens=list(text)
print(len(tokens))
print(tokens)

80
['T', 'h', 'e', ' ', 'i', 'n', 'f', 'e', 'r', 'e', 'n', 'c', 'e', ' ', 'e', 'n', 'g', 'i', 'n', 'e', ' ', 'p', 'r', 'o', 'c', 'e', 's', 's', 'e', 's', ' ', 'r', 'e', 'q', 'u', 'e', 's', 't', 's', ' ', 'b', 'y', ' ', 'l', 'o', 'a', 'd', 'i', 'n', 'g', ' ', 'm', 'o', 'd', 'e', 'l', ' ', 'w', 'e', 'i', 'g', 'h', 't', 's', ' ', 'i', 'n', 't', 'o', ' ', 'G', 'P', 'U', ' ', 'm', 'e', 'm', 'o', 'r', 'y']


In [20]:
#count_pairs

def get_pairs(tokens):
    pairs={}
    for i in range(len(tokens)-1):
        pair=(tokens[i],tokens[i+1])
        if pair not in pairs:
            pairs[pair]=0
        pairs[pair]+=1
    return pairs

In [21]:
a = get_pairs(tokens)
sorted_a = dict(sorted(a.items(), key=lambda x: (-x[1], x[0])))
print(sorted_a)

{('i', 'n'): 4, ('e', ' '): 3, ('e', 's'): 3, ('s', ' '): 3, (' ', 'i'): 2, (' ', 'm'): 2, ('c', 'e'): 2, ('e', 'n'): 2, ('m', 'o'): 2, ('n', 'g'): 2, ('r', 'e'): 2, ('t', 's'): 2, (' ', 'G'): 1, (' ', 'b'): 1, (' ', 'e'): 1, (' ', 'l'): 1, (' ', 'p'): 1, (' ', 'r'): 1, (' ', 'w'): 1, ('G', 'P'): 1, ('P', 'U'): 1, ('T', 'h'): 1, ('U', ' '): 1, ('a', 'd'): 1, ('b', 'y'): 1, ('d', 'e'): 1, ('d', 'i'): 1, ('e', 'i'): 1, ('e', 'l'): 1, ('e', 'm'): 1, ('e', 'q'): 1, ('e', 'r'): 1, ('f', 'e'): 1, ('g', ' '): 1, ('g', 'h'): 1, ('g', 'i'): 1, ('h', 'e'): 1, ('h', 't'): 1, ('i', 'g'): 1, ('l', ' '): 1, ('l', 'o'): 1, ('m', 'e'): 1, ('n', 'c'): 1, ('n', 'e'): 1, ('n', 'f'): 1, ('n', 't'): 1, ('o', ' '): 1, ('o', 'a'): 1, ('o', 'c'): 1, ('o', 'd'): 1, ('o', 'r'): 1, ('p', 'r'): 1, ('q', 'u'): 1, ('r', 'o'): 1, ('r', 'y'): 1, ('s', 'e'): 1, ('s', 's'): 1, ('s', 't'): 1, ('t', 'o'): 1, ('u', 'e'): 1, ('w', 'e'): 1, ('y', ' '): 1}


In [22]:
#Merge_Pairs
def merge_pair(tokens, pair):
    result = []
    i = 0

    while i < len(tokens):
        has_next = i + 1 < len(tokens)
        current = (tokens[i], tokens[i+1]) if has_next else None

        if current == pair:
            result.append(tokens[i] + tokens[i+1])
            i += 2  # two tokens consumed
        else:
            result.append(tokens[i])
            i += 1  # only one consumed

    return result


print(merge_pair(tokens, ("e", "s"))) #The e,s will get together

['T', 'h', 'e', ' ', 'i', 'n', 'f', 'e', 'r', 'e', 'n', 'c', 'e', ' ', 'e', 'n', 'g', 'i', 'n', 'e', ' ', 'p', 'r', 'o', 'c', 'es', 's', 'es', ' ', 'r', 'e', 'q', 'u', 'es', 't', 's', ' ', 'b', 'y', ' ', 'l', 'o', 'a', 'd', 'i', 'n', 'g', ' ', 'm', 'o', 'd', 'e', 'l', ' ', 'w', 'e', 'i', 'g', 'h', 't', 's', ' ', 'i', 'n', 't', 'o', ' ', 'G', 'P', 'U', ' ', 'm', 'e', 'm', 'o', 'r', 'y']


In [29]:
def train_bpe(text, num_merges=200):
    tokens = list(text)
    learned_merges = []

    for step in range(num_merges):
        pairs = get_pairs(tokens)
        if not pairs:
            break

        # Find the pair with the maximum frequency using standard dict operations
        # Resolves tie-breaking consistently
        best_pair = max(pairs, key=pairs.get)
        count = pairs[best_pair]

        tokens = merge_pair(tokens, best_pair)
        learned_merges.append(best_pair)

        print(
            f"Step {step + 1}: {best_pair} appeared {count} times "
            f"-> {len(tokens)} tokens remain"
        )

    vocab = {
        token: token_id
        for token_id, token in enumerate(sorted(set(tokens)))
    }

    return tokens, learned_merges, vocab

# Call the function to see the print output
tokens_out, merges_out, vocab_out = train_bpe(text)

Step 1: ('i', 'n') appeared 4 times -> 76 tokens remain
Step 2: ('e', ' ') appeared 3 times -> 73 tokens remain
Step 3: ('e', 's') appeared 3 times -> 70 tokens remain
Step 4: ('r', 'e') appeared 2 times -> 68 tokens remain
Step 5: ('t', 's') appeared 2 times -> 66 tokens remain
Step 6: ('ts', ' ') appeared 2 times -> 64 tokens remain
Step 7: (' ', 'm') appeared 2 times -> 62 tokens remain
Step 8: ('T', 'h') appeared 1 times -> 61 tokens remain
Step 9: ('Th', 'e ') appeared 1 times -> 60 tokens remain
Step 10: ('The ', 'in') appeared 1 times -> 59 tokens remain
Step 11: ('The in', 'f') appeared 1 times -> 58 tokens remain
Step 12: ('The inf', 'e') appeared 1 times -> 57 tokens remain
Step 13: ('The infe', 're') appeared 1 times -> 56 tokens remain
Step 14: ('The infere', 'n') appeared 1 times -> 55 tokens remain
Step 15: ('The inferen', 'c') appeared 1 times -> 54 tokens remain
Step 16: ('The inferenc', 'e ') appeared 1 times -> 53 tokens remain
Step 17: ('The inference ', 'e') appeare

In [27]:
def tokenize(text, learned_merges):
    tokens = list(text)

    # Replay the merge rules in the same order used during training.
    for pair in learned_merges:
        tokens = merge_pair(tokens, pair)

    return tokens

In [31]:
training_text = "The sky is blue"

final_tokens, learned_merges, vocab = train_bpe(
    training_text,
    num_merges=8,
)

print("\nLearned merges:", learned_merges)
print("Vocabulary:", vocab)
print("sky  ->", tokenize("sky", learned_merges))
print("blue ->", tokenize("blue", learned_merges))


Step 1: ('T', 'h') appeared 1 times -> 14 tokens remain
Step 2: ('Th', 'e') appeared 1 times -> 13 tokens remain
Step 3: ('The', ' ') appeared 1 times -> 12 tokens remain
Step 4: ('The ', 's') appeared 1 times -> 11 tokens remain
Step 5: ('The s', 'k') appeared 1 times -> 10 tokens remain
Step 6: ('The sk', 'y') appeared 1 times -> 9 tokens remain
Step 7: ('The sky', ' ') appeared 1 times -> 8 tokens remain
Step 8: ('The sky ', 'i') appeared 1 times -> 7 tokens remain

Learned merges: [('T', 'h'), ('Th', 'e'), ('The', ' '), ('The ', 's'), ('The s', 'k'), ('The sk', 'y'), ('The sky', ' '), ('The sky ', 'i')]
Vocabulary: {' ': 0, 'The sky i': 1, 'b': 2, 'e': 3, 'l': 4, 's': 5, 'u': 6}
sky  -> ['s', 'k', 'y']
blue -> ['b', 'l', 'u', 'e']
